# EvoAI Lab — Training Notebook
## "Most AI training asks if the model is right or wrong. We train for something harder: is it right about being right?"

This notebook trains Llama-3-8B using the EvoAI Lab self-improving loop.
The training target is **calibration** — eliminating Zone C (high confidence + wrong answer).

In [ ]:
!pip install openenv groq sentence-transformers faiss-cpu httpx trl transformers unsloth fastapi uvicorn pydantic datasets accelerate -q

In [ ]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [ ]:
!git clone https://github.com/YOUR_USERNAME/evoai-lab.git
%cd evoai-lab
from backend.env.evoai_env import EvoAIEnv
from backend.core.pipeline import EvoAIPipeline

In [ ]:
env = EvoAIEnv()
env.reset()
results = []
for i in range(50):
    result = await env.step()
    results.append(result)
    obs = result["observation"]
    probe = obs.get("last_probe", {})
    print(f"Step {i+1}: reward={result['reward']:.3f} zone={probe.get('zone','?')}")
env.pipeline.dataset_builder.flush_to_disk()

In [ ]:
import os
os.makedirs("plots", exist_ok=True)
import matplotlib.pyplot as plt
rewards = [r["reward"] for r in results]
plt.figure(figsize=(10, 4))
plt.plot(rewards, color="#1D9E75", linewidth=1.5)
plt.axhline(0, color="#E24B4A", linewidth=0.8, linestyle="--", label="neutral")
plt.xlabel("Training step")
plt.ylabel("Reward")
plt.title("EvoAI Lab — dual-axis reward curve")
plt.legend()
plt.tight_layout()
plt.savefig("plots/reward_curve.png", dpi=150)
plt.show()
print(f"Mean reward: {sum(rewards)/len(rewards):.3f}")
print(f"Zone C count start: {results[0]['observation']['zone_c_count']}")
print(f"Zone C count end:   {results[-1]['observation']['zone_c_count']}")

In [ ]:
!python backend/training/train_grpo.py

In [ ]:
import pandas as pd
final_state = env.state()
nodes = final_state["calibration_map"]["nodes"]
df = pd.DataFrame(nodes)
df["zone_colour"] = df["zone"].map({"zone_c": "🔴", "zone_b": "🟡", "green": "🟢"})
print(df[["zone_colour", "topic", "question_type", "difficulty_tier", "confidence_avg", "visit_count"]].to_string(index=False))